# Kiwi BM25 Retriever — 한국어 최적화 키워드 검색

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **kiwipiepy** 형태소 분석기를 사용해 한국어 텍스트를 토큰화하는 방법 익히기
- `BM25Retriever`의 `preprocess_func`에 Kiwi 토큰화 함수를 연결하는 패턴 이해하기
- 일반 BM25 vs Kiwi BM25 vs FAISS vs Ensemble 검색기의 **한국어 검색 품질 차이** 비교하기
- `BaseRetriever`를 상속해 **KiwiBM25Retriever 클래스를 직접 구현**하기

## 사전 지식

- BM25: TF-IDF를 개선한 키워드 빈도 기반 검색 알고리즘
- 형태소 분석(Morphological Analysis): 단어를 의미 있는 최소 단위로 분리
- EnsembleRetriever: 여러 검색기를 가중 결합하는 하이브리드 검색

---

## 왜 Kiwi BM25인가?

### 일반 BM25의 한계 (한국어)

한국어는 조사와 어미가 단어에 붙어서 공백 기준 분리만으로는 정확한 매칭이 어려워요:

```
검색어: "금융보험"
일반 BM25 (공백 분리): "금융보험" 전체가 하나의 토큰 → "금융보험은"과 매칭 실패
Kiwi BM25 (형태소 분리): "금융" + "보험" → "금융보험은"에서도 정확히 매칭
```

### Kiwi BM25의 장점

| 기능 | 설명 |
|------|------|
| 형태소 분석 | 명사, 동사, 형용사 등 의미 단위 추출 |
| 불용어 처리 | 조사, 어미 등이 별도 토큰으로 분리되어 매칭에 영향 최소화 |
| 복합어 처리 | "금융저축보험" → "금융" + "저축" + "보험" 분리 |

```bash
pip install kiwipiepy rank_bm25
```

> **실무 팁**: KiwiBM25를 11번 노트북의 EnsembleRetriever와 결합하면 한국어 문서에서 더욱 강력한 하이브리드 검색이 가능해요.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Kiwi 형태소 분석기 기초

[kiwipiepy](https://github.com/bab2min/kiwipiepy)는 한국어 형태소 분석기 Kiwi의 Python 래퍼예요. 빠른 속도와 높은 정확도로 한국어 텍스트를 형태소 단위로 분리해줘요.

In [ ]:
from kiwipiepy import Kiwi


# 형태소 분석 예시

# 아래에 코드를 작성하세요


## BM25 + Kiwi 토큰화 함수 적용

`BM25Retriever`는 `preprocess_func` 매개변수를 제공해요. 이 함수에 Kiwi 토큰화 함수를 전달하면, 색인과 검색 모두에서 형태소 단위 매칭이 이루어져요.

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# 한국어 검색 품질 테스트용 문서 — 유사한 단어가 다르게 조합된 문서들


# Kiwi 토큰화 함수: BM25Retriever의 preprocess_func에 전달
def kiwi_tokenize(text):
    return [token.form for token in kiwi.tokenize(text)]
# 아래에 코드를 작성하세요


In [ ]:
# 각 문서의 Kiwi 토큰화 결과 확인

# 아래에 코드를 작성하세요


## 검색기 비교 실험

다음 5가지 검색기를 동일한 쿼리로 비교해볼게요:

| 검색기 | 토큰화 방식 | 특징 |
|--------|------------|------|
| **BM25** | 공백 분리 | 한국어에서 조사/어미 포함 매칭 |
| **Kiwi BM25** | 형태소 분리 | 의미 단위 정확 매칭 |
| **FAISS** | 임베딩 벡터 | 의미 유사도 기반 검색 |
| **Ensemble (7:3)** | Kiwi BM25 70% + FAISS 30% | 키워드 중심 하이브리드 |
| **Ensemble (3:7)** | Kiwi BM25 30% + FAISS 70% | 의미 중심 하이브리드 |

In [ ]:
import os


from langchain.retrievers import EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# 1) 일반 BM25: 공백 기반 분리

# 2) Kiwi BM25: 형태소 분석 기반 분리

# 3) FAISS: 임베딩 기반 의미 검색

# 4) Ensemble: Kiwi BM25 + FAISS (키워드 70% + 의미 30%)

# 5) Ensemble: Kiwi BM25 + FAISS (키워드 30% + 의미 70%)

# 아래에 코드를 작성하세요


In [ ]:
def print_search_results(retrievers, query):
    """여러 검색기의 결과를 비교 출력"""
    print(f"📝 Query: {query}")
    for name, retriever in retrievers.items():
        result = retriever.invoke(query)
        print(f"  {name:30s} → {result[0].page_content[:60]}...")
    print("=" * 80)


# 핵심 비교 쿼리: 일반 BM25의 한계가 드러나는 케이스들

# 아래에 코드를 작성하세요


### 검색 결과 분석

| 쿼리 | 일반 BM25 문제 | Kiwi BM25 개선 |
|------|---------------|----------------|
| "금융보험" | "금융보씨..." 매칭 (오탐) | "금융" + "보험" 정확 매칭 |
| "금융저축보험" | "금융보씨..." 매칭 (오탐) | "금융" + "저축" + "보험" 분리 매칭 |
| "저축금융보험" | "금융보씨..." 매칭 (오탐) | 어순 무관 형태소 매칭 |

> **실무 팁**: Ensemble 검색기에서 Kiwi BM25 비중을 높이면(7:3) 키워드 정확도가 올라가고, FAISS 비중을 높이면(3:7) 의미적 유사도가 반영돼요. 한국어 RAG에서는 **Kiwi BM25 70% + FAISS 30%** 조합이 좋은 시작점이에요.

## KiwiBM25Retriever 직접 구현

앞에서 `BM25Retriever(preprocess_func=kiwi_tokenize)` 패턴을 배웠어요. 이제 이 패턴을 **재사용 가능한 클래스**로 만들어 볼게요.

`BaseRetriever`를 상속하고 `rank_bm25.BM25Okapi`를 내부에 사용하면, Kiwi 형태소 분석이 기본 내장된 Retriever를 직접 만들 수 있어요.

In [ ]:
from __future__ import annotations

from typing import Any, Callable, Dict, Iterable, List, Optional

import numpy as np
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from pydantic import Field
from rank_bm25 import BM25Okapi


# 기본 전처리 함수: Kiwi 형태소 분석 기반 토큰화
def kiwi_preprocessing_func(text: str) -> List[str]:
    return [token.form for token in kiwi.tokenize(text)]


class KiwiBM25Retriever(BaseRetriever):
    """Kiwi 형태소 분석 기반 BM25 Retriever.



    class Config:
        arbitrary_types_allowed = True

    def from_documents(
        cls,
        documents: Iterable[Document],
        *,
        bm25_params: Optional[Dict[str, Any]] = None,
        preprocess_func: Callable[[str], List[str]] = kiwi_preprocessing_func,
        **kwargs: Any,
    ) -> KiwiBM25Retriever:
        """Document 리스트로부터 KiwiBM25Retriever를 생성해요."""
        texts, metadatas = zip(*((d.page_content, d.metadata) for d in documents))
        texts_processed = [preprocess_func(t) for t in texts]
        bm25_params = bm25_params or {}
        vectorizer = BM25Okapi(texts_processed, **bm25_params)
        docs = [Document(page_content=t, metadata=m) for t, m in zip(texts, metadatas)]
        return cls(
            vectorizer=vectorizer,
            docs=docs,
            preprocess_func=preprocess_func,
            **kwargs,
        )

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        """쿼리를 형태소 분석 후 BM25로 관련 문서를 검색해요."""
        processed_query = self.preprocess_func(query)
        return self.vectorizer.get_top_n(processed_query, self.docs, n=self.k)

# 아래에 코드를 작성하세요


In [ ]:
# KiwiBM25Retriever 사용 테스트

# 검색 테스트: 일반 BM25에서 오탐이 발생하던 쿼리

# 아래에 코드를 작성하세요


## 참고: Konlpy 형태소 분석기 비교

Kiwi 외에도 한국어 형태소 분석기로 [KoNLPy](https://konlpy.org/) 패키지의 Okt, Komoran 등을 사용할 수 있어요. 각 분석기마다 토큰화 결과가 다르므로, 도메인에 맞는 분석기를 선택하는 것이 중요해요.

> **실무 팁**: Kiwi는 설치가 간편하고(`pip install kiwipiepy`) Java 의존성이 없어서 실무에서 선호돼요. KoNLPy는 Java(JDK)가 필요해요.

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | Kiwi 형태소 분석으로 한국어 토큰화 → BM25 키워드 검색 품질 개선 |
| 간편 패턴 | `BM25Retriever.from_documents(docs, preprocess_func=kiwi_tokenize)` |
| 커스텀 클래스 | `BaseRetriever` 상속 + `BM25Okapi` 내장 → `KiwiBM25Retriever` 직접 구현 |
| 설치 | `pip install kiwipiepy rank_bm25` |
| 일반 BM25와 차이 | 한국어 어절 단위가 아닌 형태소 단위로 색인 — 어미/조사 변형 처리 |
| 최적 사용 | 단독보다 FAISS와 EnsembleRetriever로 결합 시 효과 극대화 |

### 한국어 BM25 비교

| 방식 | 토큰화 기준 | 한국어 처리 | 권장 상황 |
|------|-------------|-------------|-----------|
| 일반 BM25 | 공백 분리 | 조사·어미 포함 색인 (오탐 위험) | 영어 중심 문서 |
| Kiwi BM25 (`preprocess_func`) | 형태소 분리 | 의미 형태소 단위 색인 | 한국어 문서 (빠른 적용) |
| KiwiBM25Retriever (직접 구현) | 형태소 분리 (내장) | 커스텀 Retriever 클래스 | 한국어 문서 (재사용 가능) |

### 다음 노트북 예고

**11-CC-EnsembleRetriever** — Contextual Compression(문맥 압축)과 EnsembleRetriever를 결합해 이 시리즈 최고 수준의 검색 파이프라인을 구성하는 방법을 배워요.

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | Kiwi 형태소 분석으로 한국어 토큰화 → BM25 키워드 검색 품질 개선 |
| 핵심 패턴 | `BM25Retriever.from_documents(docs, preprocess_func=kiwi_tokenize)` |
| 래퍼 클래스 | `langchain_teddynote.retrievers.KiwiBM25Retriever` |
| 설치 | `pip install kiwipiepy rank_bm25 langchain-teddynote` |
| 일반 BM25와 차이 | 한국어 어절 단위가 아닌 형태소 단위로 색인 — 어미/조사 변형 처리 |
| 최적 사용 | 단독보다 FAISS와 EnsembleRetriever로 결합 시 효과 극대화 |

### 한국어 BM25 비교

| 방식 | 토큰화 기준 | 한국어 처리 | 권장 상황 |
|------|-------------|-------------|-----------|
| 일반 BM25 | 공백 분리 | 조사·어미 포함 색인 (오탐 위험) | 영어 중심 문서 |
| Kiwi BM25 | 형태소 분리 | 의미 형태소 단위 색인 | 한국어 문서 |
| KiwiBM25Retriever | 형태소 분리 (내장) | Kiwi BM25 + 편의 API | 한국어 문서 (간편 사용) |

### 다음 노트북 예고

**11-CC-EnsembleRetriever** — Contextual Compression(문맥 압축)과 EnsembleRetriever를 결합해 이 시리즈 최고 수준의 검색 파이프라인을 구성하는 방법을 배워요.